#Enterprise Fleet Analytics Pipeline: Focuses on the business outcome (analytics) and the domain (fleet/logistics).

![logistics](logistics_project.png)

Download the data from the below gdrive and upload into the catalog
https://drive.google.com/drive/folders/1J3AVJIPLP7CzT15yJIpSiWXshu1iLXKn?usp=drive_link

##**1. Data Munging** -

####1. Visibily/Manually opening the file and capture couple of data patterns (Manual Exploratory Data Analysis)


####EDA of the file - logistics_shipment_detail_3000
- it is a semi structured json file
- no header,footer and comment
- data quality<br>
        > data types are mapped correctly<br>
        > data set has expected spark date format<br>
        > no nulls and duplication<br>
        > data are uniform <br>
####EDA of the file - logistics_source1
- it is a structured csv file
- header provided
- no footer and comment
- presence of null column and null record
- data quality<br>
    > age has string value "ten" - (reference- ship_id -5000006)<br>
    > shipment_id has data format issue ("ten is mentioned")<br>
    > extra column ("additional column " ship_id-5000006)<br>
    >count of rows having less column- 4<br>
####EDA of the file - logistics_source2
- it is a structured csv file
- header provided
- no footer and comment
- presence of null column and null record
- data quality<br>
    > age has string value "ten"<br>
    > format issue : no comma provided (6000003,Michael,Chen,41,Warehouse Manager,Singapore,)<br>
    > shipment_id has data format issue ("ten is mentioned")<br>
    > count of rows having less column- 5<br>
    





####2. Programatically finding couple of data patterns applying below EDA (File: logistics_source1)


In [0]:
#Create a Spark Session Object
from pyspark.sql import SparkSession
spark= SparkSession.builder.appName("fleet pipeline").getOrCreate()

In [0]:
#Analysing the schema, datatypes, columns etc.,
df= spark.read.csv("/Volumes/data_catalog/data_schema/data_volume/logistics_source1",header=True,inferSchema=True)
df.printSchema()
df_schema="shipment_id long, first_name string, last_name string, age long, role string"
#since schme acan be applied only while reading data ,we are re-reading the data
df= spark.read.csv("/Volumes/data_catalog/data_schema/data_volume/logistics_source1",header=True,inferSchema=True,schema=df_schema)
df.printSchema()
display(df)
#Analysing the duplicate records count and summary of the dataframe.
display(df.distinct())
##--------- 1 duplicate record is found in total---------
display(df.summary())




####capturings
shipment_id - 2 nulls <br>
first_name  - 3 nulls <br>
last_name,age,role   - 4 nulls <br>

###**b. Active Data Munging** File: logistics_source1 and logistics_source2

#####1.Combining Data + Schema Merging (Structuring)


In [0]:
# reading the files from the source
from pyspark.sql.functions import *
log1_df=spark.read.csv("/Volumes/data_catalog/data_schema/data_volume/logistics_source1",header=True,inferSchema=True)
log2_df=spark.read.csv("/Volumes/data_catalog/data_schema/data_volume/logistics_source2",header=True,inferSchema=True)
log1_df=log1_df.withColumn("source",lit("source1"))
log2_df=log2_df.withColumn("source",lit("source2"))
merged_df=log1_df.unionByName(log2_df, allowMissingColumns=True)
merged_df.printSchema()
display(merged_df)
#--------using unionByname as the columns are of different order in both the files,safe for schmea evolution and for the production---------


#####2. Cleansing, Scrubbing: 


In [0]:
# Removal of null records in the dataset - cleansing

print("---------------------removing the null records in the crucial columns------------------")
merged_df = merged_df.na.drop(
    how='any',
    subset=["shipment_id", "role"]
)
# Drop rows if both first_name and last_name are null
merged_df = merged_df.na.drop(
    how='all',
    subset=["first_name", "last_name"]
)

# Scrubbing: fill -1 in null fields of age and 'UNKNOWN' in vehicle_type - this is a default rule
merged_df = merged_df.na.fill(
    {
        "age": -1,
        "vehicle_type": "UNKNOWN"
    }
)

# replacing inconsisitent data values  in vehicle types: truck to LMV bike to TwoWheeler and in age column ten-> 10
merged_df=merged_df.replace({"Truck":"LMV","Bike":"Two Wheeler"},subset=["vehicle_type"])
scrubbed_df=merged_df.replace({"ten":"10"},subset=["age"])
display(scrubbed_df)




####3. Standardization, De-Duplication and Replacement / Deletion of Data to make it in a usable format

Detail Dataframe creation <br>
1. Read Data from logistics_shipment_detail.json
2. As this data is a clean json data, it doesn't require any cleansing or scrubbing.

In [0]:
#standardisation of data fields
#column uniformity by altering the cases
from pyspark.sql.types import *
df_json=spark.read.json("/Volumes/data_catalog/data_schema/data_volume/logistics_shipment_detail_3000.json",multiLine=True)
df_json= df_json.withColumn("domain",lit("Logistics")).withColumn("vehicle_type",upper("vehicle_type")).withColumn("ingestion_timestamp",current_timestamp()).withColumn("is_expedited",lit("False"))
display(df_json)
scrubbed_df=scrubbed_df.withColumns({"role":lower("role"),"hub_location":initcap("hub_location")})

display(scrubbed_df)

#column formating
#padding with 0's for reporting and other analaysis to the shippment_id
df_json = df_json.withColumn("shipment_id", lpad(col("shipment_id").cast("string"), 10, "0"))
display(df_json)
df_json=df_json.withColumn("shipment_date",to_date(col("shipment_date"))).withColumn("shipment_cost",col("shipment_cost").cast(DecimalType(7,2)))
display(df_json)

#standardisation of data fields to a respective data format
scrubbed_df= scrubbed_df.withColumn("age",col("age").cast("int"))
scrubbed_df.printSchema()
df_json= df_json.withColumn("shipment_weight_kg",col("shipment_weight_kg").cast("double"))
#.withColumn("is_expedited",col("is_expedited").cast("boolean"))

scrubbed_df=scrubbed_df.withColumnsRenamed({"first_name":"staff_first_name","last_name":"staff_last_name","hub_location":"origin_hub_city"})
display(scrubbed_df)

#shipment_id (Identifier), staff_first_name (Dimension)staff_last_name (Dimension), role (Dimension), origin_hub_city (Location), shipment_cost (Metric), ingestion_timestamp (Audit)

standard_df=df_json.unionByName(scrubbed_df, allowMissingColumns=True)

standard_dfnew=standard_df.select("shipment_id","staff_first_name","staff_last_name","role","origin_hub_city","shipment_cost","shipment_date","ingestion_timestamp")
display(standard_dfnew)





Deduplication:
1. Apply Record Level De-Duplication
2. Apply Column Level De-Duplication (Primary Key Enforcement)

In [0]:
#deduplication of records in the dataframe
standard_df= standard_df.distinct()
display(standard_df)
#deduplication of records in column level
#primary key enforcement - Uniqueness,not Null
standard_df=standard_df.dropDuplicates(subset=["shipment_id"])
display(standard_df)

##2. Data Enrichment - Detailing of data
Makes your data rich and detailed <br>

In [0]:
# for auditing purposes enriching the dataset by capturing the load_dt
scrubbed_df= scrubbed_df.withColumns({"load_dt":current_timestamp(),"full_name":concat_ws(" ",col("staff_first_name"),col("staff_last_name"))})
standard_df= standard_df.withColumn("route_segment",concat_ws("-","source_city","destination_city")).withColumn("vehicle_identifier",concat_ws("_","vehicle_type","shipment_id"))
display(scrubbed_df)
display(standard_df)

In [0]:
#Deriving of Columns (Time Intelligence)
# for conditional logic use when(cond,t/f).otherwise(if fails)
df_json_enriched=df_json.withColumn("shipment_year",year(col("shipment_date"))).\
withColumn("shipment_month",month(col("shipment_date"))).\
withColumn("is_weekend",when(dayofweek(col("shipment_date")).isin(7,1),True).otherwise(False)).\
withColumn("is_expedited",when(col("shipment_status").isin("IN_TRANSIT","DELIVERED"),True).otherwise(False))

#addition of calculated_fields
df_json_enriched=df_json_enriched.withColumn("cost_per_kg",round(col("shipment_cost")/col("shipment_weight_kg"),3)).\
    withColumn("days_since_shipment",datediff(current_date(),"shipment_date")).\
    withColumn("tax_amount",round(col("shipment_cost")*0.18,3))


#splitting or merging of columns,index starts from 0
df_json_enriched=df_json_enriched.withColumn("route_lane",concat_ws("->","source_city","destination_city")).withColumn("order_sequence",substring(col("order_id"),4,7))
display(df_json_enriched)

#Remove/Eliminate (drop, select, selectExpr)
#Excluding unnecessary or redundant columns to optimize storage and privacy.

scrubbed_df=scrubbed_df.drop("staff_first_name","staff_last_name")
#or
scrubbed_df=scrubbed_df.select("shipment_id","full_name","age","role","source","origin_hub_city","load_dt")
#or
scrubbed_df=scrubbed_df.selectExpr("shipment_id","full_name as Full_name","age","role","source","origin_hub_city","load_dt")
scrubbed_df.show(5)


## 3. Data Customization & Processing - Application of Tailored Business Specific Rules



In [0]:
#UDF1: Complex Incentive Calculation
def calculate_bonus(role, age):
    if role=="driver" and age>40:
       return "15 % of salary"
    elif role=="driver" and age<30:
        return "5 % of salary"
    else:
        return 0

bonus_udf=udf(calculate_bonus,StringType())
enriched_df=scrubbed_df.withColumn("Perfromance_bonus",bonus_udf(col("role"),col("age")))

#udf2:(Privacy Compliance)
#hiding the full identity of the staff to comply with privacy laws (GDPR/DPDP), while keeping names recognizable for internal managers.
def mask_identity(name):
    return name[0:2]+"*"*len(name[2:-1])+name[-1]

mask_udf=udf(mask_identity,StringType())
enriched_df=enriched_df.withColumn("full_name",mask_udf(col("full_name")))
display(enriched_df)




## 4. Data Core Curation & Processing (Pre-Wrangling)
*Applying business logic to focus, filter, and summarize data before final analysis.*

**1. Select (Projection)**<br>

**2. Filter (Selection)**<br>

**3. Derive Flags & Columns (Business Logic)**<br>

**4. Format (Standardization)**<br>

**5. Group & Aggregate (Summarization)**<br>

**6. Sorting (Ordering)**<br>

**7. Limit (Top-N Analysis)**<br>

In [0]:
#data curation
#selecting only the necessary columns to avoid displaying of sensitive data
selected_df=scrubbed_df.select("full_name","role","origin_hub_city")
#report on active operational problems.
filtered_df=df_json_enriched.filter(col("shipment_status").isin("'DELAYED','RETURNED'"))
#Insurance audit for senior staff.
filtered_df=scrubbed_df.filter(col("age")>50)
#3. Derive Flags & Columns (Business Logic)
flag_df=df_json_enriched.withColumn("is_high_value",when(col("shipment_cost")<50,True).otherwise(False))
#Format (Standardization)
formatted_df=df_json_enriched.withColumn("shipment_cost",concat(lit("₹"),"shipment_cost")).withColumn("source_city",upper(col("source_city")))
#Group & Aggregate (Summarization)
group_df=scrubbed_df.groupBy("origin_hub_city").count()
agg_df=df_json_enriched.groupBy("vehicle_type").sum("shipment_weight_kg")
display(agg_df)
#6. Sorting (Ordering)
sort_df=df_json_enriched.orderBy("shipment_cost",ascending=False)
sort_df=df_json_enriched.orderBy("shipment_date",ascending=True)
#display(sort_df)
#7. Limit (Top-N Analysis)
top_df=df_json_enriched.filter(col("shipment_status")=="DELIVERED").orderBy("shipment_cost").limit(10)
display(top_df)


## 5. Data Wrangling - Transformation & Analytics
*Combining, modeling, and analyzing data to answer complex business questions.*

### **1. Joins**
Source Files:<br>
Left Side (staff_df):<br> logistics_source1 & logistics_source2<br>
Right Side (shipments_df):<br> logistics_shipment_detail_3000.json<br>
#### **1.1 Frequently Used Simple Joins (Inner, Left)**
* **Inner Join (Performance Analysis):**
  * **Scenario:** We only want to analyze *completed work*. Connect Staff to the Shipments they handled.
  * **Action:** Join `staff_df` and `shipments_df` on `shipment_id`.
  * **Result:** Returns only rows where a staff member is assigned to a valid shipment.
* **Left Join (Idle Resource check):**
  * **Scenario:** Find out which staff members are currently *idle* (not assigned to any shipment).
  * **Action:** Join `staff_df` (Left) with `shipments_df` (Right) on `shipment_id`. Filter where `shipments_df.shipment_id` is NULL.

#### **1.2 Infrequent Simple Joins (Self, Right, Full, Cartesian)**
* **Self Join (Peer Finding):**
  * **Scenario:** Find all pairs of employees working in the same `hub_location`.
  * **Action:** Join `staff_df` to itself on `hub_location`, filtering where `staff_id_A != staff_id_B`.
* **Right Join (Orphan Data Check):**
  * **Scenario:** Identify shipments in the system that have *no valid driver* assigned (Data Integrity Issue).
  * **Action:** Join `staff_df` (Left) with `shipments_df` (Right). Focus on NULLs on the left side.
* **Full Outer Join (Reconciliation):**
  * **Scenario:** A complete audit to find *both* idle drivers AND unassigned shipments in one view.
  * **Action:** Perform a Full Outer Join on `shipment_id`.
* **Cartesian/Cross Join (Capacity Planning):**
  * **Scenario:** Generate a schedule of *every possible* driver assignment to *every* pending shipment to run an optimization algorithm.
  * **Action:** Cross Join `drivers_df` and `pending_shipments_df`.

#### **1.3 Advanced Joins (Semi and Anti)**
* **Left Semi Join (Existence Check):**
  * **Scenario:** "Show me the details of Drivers who have *at least one* shipment." (Standard filtering).
  * **Action:** `staff_df.join(shipments_df, "shipment_id", "left_semi")`.
  * **Benefit:** Performance optimization; it stops scanning the right table once a match is found.
* **Left Anti Join (Negation Check):**
  * **Scenario:** "Show me the details of Drivers who have *never* touched a shipment."
  * **Action:** `staff_df.join(shipments_df, "shipment_id", "left_anti")`.

### **2. Lookup**<br>
Source File: logistics_source1 and logistics_source2 (merged into Staff DF)<br>
* **Scenario:** Validation. Check if the `hub_location` in the staff file exists in the corporate `Master_City_List`.
* **Action:** Compare values against a reference list.

### **3. Lookup & Enrichment**<br>
Source File: logistics_source1 and logistics_source2 (merged into Staff DF)<br>
* **Scenario:** Geo-Tagging.
* **Action:** Lookup `hub_location` ("Pune") in a Master Latitude/Longitude table and enrich the dataset by adding `lat` and `long` columns for map plotting.

### **4. Schema Modeling (Denormalization)**<br>
Source Files: All 3 Files (logistics_source1, logistics_source2, logistics_shipment_detail_3000.json)<br>
* **Scenario:** Creating a "Gold Layer" Table for PowerBI/Tableau.
* **Action:** Flatten the Star Schema. Join `Staff`, `Shipments`, and `Vehicle_Master` into one wide table (`wide_shipment_history`) so analysts don't have to perform joins during reporting.

### **5. Windowing (Ranking & Trends)**<br>
Source Files:<br>
logistics_source2: Provides hub_location (Partition Key).<br>
logistics_shipment_detail_3000.json: Provides shipment_cost (Ordering Key)<br>
* **Scenario:** "Who are the Top 3 Drivers by Cost in *each* Hub?"
* **Action:**
  1. Partition by `hub_location`.
  2. Order by `total_shipment_cost` Descending.
  3. Apply `dense_rank()` and `row_number()
  4. Filter where `rank or row_number <= 3`.

### **6. Analytical Functions (Lead/Lag)**<br>
Source File: <br>
logistics_shipment_detail_3000.json<br>
* **Scenario:** Idle Time Analysis.
* **Action:** For each driver, calculate the days elapsed since their *previous* shipment.

### **7. Set Operations**<br>
Source Files: logistics_source1 and logistics_source2<br>
* **Union:** Combining `Source1` (Legacy) and `Source2` (Modern) into one dataset (Already done in Active Munging).
* **Intersect:** Identifying Staff IDs that appear in *both* Source 1 and Source 2 (Duplicate/Migration Check).
* **Except (Difference):** Identifying Staff IDs present in Source 2 but *missing* from Source 1 (New Hires).

### **8. Grouping & Aggregations (Advanced)**<br>
Source Files:<br>
logistics_source2: Provides hub_location and vehicle_type (Grouping Dimensions).<br>
logistics_shipment_detail_3000.json: Provides shipment_cost (Aggregation Metric).<br>
* **Scenario:** The CFO wants a subtotal report at multiple levels:
  1. Total Cost by Hub.
  2. Total Cost by Hub AND Vehicle Type.
  3. Grand Total.
* **Action:** Use `cube("hub_location", "vehicle_type")` or `rollup()` to generate all these subtotals in a single query.

####supported joins 
**'inner', 'outer', 'full', 'fullouter', 'full_outer', 'leftouter', 'left', 'left_outer', 'rightouter', 'right', 'right_outer', 'leftsemi', 'left_semi', 'semi', 'leftanti', 'left_anti', 'anti', 'cross'**

In [0]:
#leftside-merged_df
#rightside-df_json
#inner join
merged_df=merged_df.withColumn("full_name",concat_ws(" ","first_name","last_name"))
mergednew_df=merged_df.select("shipment_id","full_name","age","role","source","hub_location","vehicle_type")
inner_df=merged_df.join(df_json,how="inner",on="shipment_id")
left_df=merged_df.join(df_json,how="left",on="shipment_id")#.filter(col("shipment_id").isNull()) - answer is empty becoz shipment_id column is not null so no rows displayed
#Self Join (Peer Finding)-employess in same hub location
self_join=mergednew_df.alias("m").join(mergednew_df.alias("j"),col("m.hub_location")==col("j.hub_location"),"inner").filter(col("m.full_name")!=col("j.full_name"))
#Right Join (Orphan Data Check): no driver reference in the shipments_df so considered to be all shipemnts has invalid driver
right_join=merged_df.join(df_json,how="right",on="shipment_id")
fullouter_join=merged_df.join(df_json,how="full_outer",on="shipment_id")
#cross join- does not support on condition
drivers_df = merged_df.filter(col("role") == "Driver")
delayed_shipments_df = df_json.filter(col("shipment_status") == "DELAYED")
cross_join = drivers_df.join(delayed_shipments_df,how="cross")
display(cross_join)
#Left Semi Join (Existence Check):
#LEFT SEMI JOIN returns rows from the LEFT table,ONLY IF a matching row exists in the RIGHT table.LEFT SEMI = filter left table by existence in right table
semi_lefftjoin=merged_df.join(df_json,how="left_semi",on="shipment_id")
#Left Anti Join (Non-Existence Check)
#LEFT ANTI JOIN returns rows from the LEFT table that do NOT have any matching row in the RIGHT table.
anti_leftjoin=merged_df.join(df_json,how="left_anti",on="shipment_id")

#============================================================================================================================================


# LOOKUP AND ENRICHMENT
master_df=spark.read.csv("/Volumes/data_catalog/data_schema/data_volume/Master_City_List.csv",header=True,inferSchema=True)
#Checking if the hub_location in the staff file exists in the dataframe of corporate Master_City_List.csv.
lookup_df=merged_df.join(master_df,merged_df.hub_location==master_df.city_name,'left_semi')

#geo tagging- Lookup hub_location (eg. "Pune") in a Master Latitude/Longitude Master_City_List.csv dataframe and enrich our logistics_source
lookup_enrich_df=merged_df.join(master_df,merged_df.hub_location==master_df.city_name,'inner')
display(lookup_enrich_df)

#=======================================================================================================================
#schema modelling
golden_table=merged_df.select("shipment_id","full_name","age","role","source","hub_location","vehicle_type").join(df_json,how="full").join(master_df,merged_df.hub_location==master_df.city_name,'inner')
#display(golden_table)

#=======================================================================================================================
#windowing,while doing windowing use the desc() or asc() function
#"Who are the Top 3 Drivers by Cost in each Hub?"
from pyspark.sql.window import Window
new_df=log2_df.join(df_json,how="full")

drivers_df=new_df.filter(col("role") == "Driver")
driver_cost_df=drivers_df.groupBy("hub_location").agg(sum("shipment_cost").alias("total_shipment_cost"))
windowSpec = Window.partitionBy("hub_location").orderBy(col("shipment_cost").desc())  
top3_drivers_df=drivers_df.withColumn("sno",row_number().over(windowSpec)).withColumn("rank",dense_rank().over(windowSpec)).filter(col("rank") <= 3)
final=top3_drivers_df.select("first_name","last_name","hub_location","shipment_cost","sno","rank")
#display(final)

#=======================================================================================================================
#Analytical Functions (Lead/Lag)

analytical_df=new_df.filter(col("role") == "Driver")
final_df=analytical_df.withColumn("prev_shipment_date",lag(col("shipment_date"),1).over(Window.partitionBy("hub_location").orderBy(col("shipment_date")))).withColumn("ELAPSED_DAYS",datediff(col("shipment_date"),col("prev_shipment_date")))
#display(final_df)

#=======================================================================================================================
#Set Operations,subtract- no of columns should be same in both the dataframes
#--Same number of columns
#--Same data types
#--Same column order, subtract-Returns rows that are present in the first dataset but NOT in the second.,intersect_all-Returns common rows including duplicates.
intersect_df=log1_df.intersect(log2_df)
#difference_df=log1_df.subtract(log2_df)
#display(difference_df)

#=======================================================================================================================
#Grouping & Aggregations (Advanced) rollup and cube-methods on a GroupedData object.
l_d=log2_df.drop("vehicle_type")
d_f=l_d.join(df_json,how="full")
new_d=d_f.select("hub_location","role","vehicle_type","shipment_cost","shipment_date")
summary_df= new_d.rollup("hub_location").agg(sum("shipment_cost")).alias("total_cost").orderBy("vehice_type",ascending=True)
summary2_df=new_d.cube("hub_location","vehicle_type").agg(sum("shipment_cost")).alias("total_cost").orderBy("sum(shipment_cost)")
display(summary2_df)



##6. Data Persistance (LOAD)-> Data Publishing & Consumption<br>

Store the inner joined, lookup and enrichment, Schema Modeling, windowing, analytical functions, set operations, grouping and aggregation data into the delta tables.

In [0]:
#standard_dfnew.write.format("delta").mode("overwrite").save("/Volumes/lakehouse/lakehouse_schema/lakehouse_volume/munged_delta")
df_json_enriched.write.format("json").mode("overwrite").save("/Volumes/lakehouse/lakehouse_schema/lakehouse_volume/enrich_json")